# Tweety-5 : Argumentation Abstraite de Dung (C# / .NET) — Tranche 1 : Moteur from-scratch

**Navigation** : [<< Tweety-4 BeliefRevision-Csharp](Tweety-4-Belief-Revision-Csharp.ipynb) | [Tweety-5 Python](Tweety-5-Abstract-Argumentation.ipynb) | [Index](../README.md)

**Twin .NET du notebook [Tweety-5-Abstract-Argumentation](Tweety-5-Abstract-Argumentation.ipynb) (Python/Tweety-JVM).** Ce notebook implemente les **semantiques de Dung from-scratch** en C#/.NET, sans librairie externe.

## Complementarite (.NET from-scratch ↔ Python/IKVM), pas workaround (#3801)

| Twin | Approche | Valeur pedagogique |
|------|----------|--------------------|
| **Python (Tweety-JVM)** | librairie Tweety (`SimpleGroundedReasoner`, CF2, equivalence) | utiliser un solveur SOTA tout pret, acceder aux semantiques avancees |
| **Tweety-3-Dung-Csharp (IKVM)** | Tweety compile via IKVM | meme lib, cote .NET |
| **.NET (this)** | **implementation from-scratch** en C# | comprendre la fonction caracteristique et l'enumeration des extensions de l'interieur |

Les semantiques de Dung (grounded, stable, preferred, complete) sont **algorithmiquement simples** (iteration de point fixe, enumeration d'ensembles) : les implementer a la main est precisement l'occasion pedagogique. Les semantiques avancees (CF2, equivalence, explications) restent l'apanage des twins librairie (tranches ulterieures).

## Objectifs d'apprentissage (tranche 1)

A la fin de cette tranche, vous saurez :
1. Modeliser un **cadre d'argumentation abstrait** (AF) de Dung (arguments + relation d'attaque)
2. Definir et implementer la **fonction caracteristique** $F(S)$ et l'**acceptabilite**
3. Calculer les semantiques **grounded** (point fixe), **stable**, **preferred**, **complete**
4. Manipuler le **labeling a trois valeurs** (in / out / undec)
5. Generer des cadres aleatoires et etudier la taille de l'extension grounded

### Prerequis
- Tweety-2 (logiques de base), Tweety-3-Dung-Csharp (vue librairie du meme sujet)
- Notions de theorie des graphes orientes et de theorie des ensembles

### Duree estimee : 50 minutes

> **Note** : L'argumentation abstraite de Dung (1995) ignore la structure interne des arguments : seul compte **qui attaque qui**. C'est une theorie remarquablement elegante — quatre semantiques differentes decident quels arguments sont "acceptables" a partir de la seule topologie du graphe d'attaque.

In [1]:
// Setup : aucune dependance externe — moteur Dung from-scratch, uniquement la BCL .NET.
using Microsoft.DotNet.Interactive;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;

// PRNG deterministe (seed fixe) pour reproductibilite pedagogique.
var rng = new Random(42);
"Environnement pret — moteur d'argumentation de Dung from-scratch (BCL .NET seule).".Display();

The below script needs to be able to find the current output cell; this is an easy method to get it.

Environnement pret — moteur d'argumentation de Dung from-scratch (BCL .NET seule).

## 1. Cadres d'argumentation abstraits (Dung 1995)

Un **cadre d'argumentation abstrait** (AF) est un couple $AF = \langle A, R \rangle$ ou :
- $A$ est un ensemble fini d'**arguments** (boites noires, on ignore leur contenu),
- $R \subseteq A \times A$ est une relation d'**attaque** : $(a, b) \in R$ se lit "$a$ attaque $b$".

On note $a \to b$. Un argument $b$ est **attaque** s'il existe $a$ avec $a \to b$. Un argument $a$ **defend** $b$ (contre l'attaquant $c$) si $a \to c$.

### Le coeur de la theorie : la fonction caracteristique

La **fonction caracteristique** $F : 2^A \to 2^A$ est definie par :

$$F(S) = \{\, a \in A \mid \forall b \text{ tel que } b \to a,\ \exists c \in S \text{ tel que } c \to b \,\}$$

$F(S)$ est l'ensemble des arguments **acceptables** relativement a $S$ : ceux dont tous les attaquants sont eux-memes attaques par un element de $S$ (i.e. **defendus** par $S$).

Les quatres semantiques classiques se construisent sur $F$ :

| Semantique | Definition |
|------------|------------|
| **Admissible** | $S$ sans conflit ET $S \subseteq F(S)$ (tout element de $S$ est defendu par $S$) |
| **Complete** | Admissible ET $F(S) \subseteq S$ (point fixe local de $F$) |
| **Grounded** | Le **plus petit** point fixe de $F$ (itere depuis $\emptyset$) — unique, toujours defini |
| **Preferred** | Une extension admissible **maximale** par inclusion — peut etre multiple |
| **Stable** | Sans conflit ET attaque tout argument hors de $S$ — peut ne pas exister |

In [2]:
// Representation d'un cadre d'argumentation abstrait (Dung AF).
// Graphe oriente : arguments (noeuds) + attaques (aretes diriges).
class DungAF
{
    public HashSet<string> Args { get; } = new();
    // attackersOf[b] = { a : a -> b }  (qui attaque b)
    // attackedBy[a] = { b : a -> b }   (qui a attaque)
    private readonly Dictionary<string, HashSet<string>> _attackersOf = new();
    private readonly Dictionary<string, HashSet<string>> _attackedBy = new();

    public void AddArg(string a)
    {
        Args.Add(a);
        if (!_attackersOf.ContainsKey(a)) _attackersOf[a] = new();
        if (!_attackedBy.ContainsKey(a)) _attackedBy[a] = new();
    }

    public void AddAttack(string a, string b)
    {
        AddArg(a); AddArg(b);
        _attackersOf[b].Add(a);
        _attackedBy[a].Add(b);
    }

    public IEnumerable<string> AttackersOf(string b) => _attackersOf[b];
    public bool Attacks(string a, string b) => _attackedBy[a].Contains(b);
    public bool HasArgs() => Args.Count > 0;

    public override string ToString()
    {
        var atks = string.Join(", ", Args.OrderBy(x => x)
            .SelectMany(a => _attackedBy[a].OrderBy(b => b).Select(b => $"{a}->{b}")));
        return $"AF(args={{{string.Join(", ", Args.OrderBy(x => x))}}}, attacks={{{atks}}})";
    }
}
"Classe DungAF prete (attackersOf / attackedBy indexes pour O(1) lookup).".Display();

Classe DungAF prete (attackersOf / attackedBy indexes pour O(1) lookup).

## 2. Acceptabilite et fonction caracteristique

L'**acceptabilite** est la brique de base : un argument $a$ est acceptable relativement a $S$ si **chaque attaquant** de $a$ est **contre-attaque** par un element de $S$. On l'implemente litteralement : pour chaque attaquant $b$ de $a$, on verifie qu'un $c \in S$ attaque $b$.

La fonction caracteristique $F(S)$ collecte tous les arguments acceptables relativement a $S$.

In [3]:
// a est-il defendu par S ? (tous ses attaquants sont contre-attaques par S)
static bool DefendedBy(string a, ISet<string> S, DungAF af)
{
    foreach (var b in af.AttackersOf(a))
    {
        bool blocked = false;
        foreach (var c in S)
            if (af.Attacks(c, b)) { blocked = true; break; }
        if (!blocked) return false;
    }
    return true;
}

// Fonction caracteristique F(S) = { a dans A : a defendu par S }
static HashSet<string> CharacteristicFunction(ISet<string> S, DungAF af)
{
    var result = new HashSet<string>();
    foreach (var a in af.Args)
        if (DefendedBy(a, S, af)) result.Add(a);
    return result;
}

// Verif sur un petit exemple : A -> B -> C -> D (chaine).
var chain = new DungAF();
chain.AddAttack("A", "B"); chain.AddAttack("B", "C"); chain.AddAttack("C", "D");
var sb = new StringBuilder();
sb.AppendLine($"Cadre : {chain}");
sb.AppendLine($"F(∅)  = {{{string.Join(", ", CharacteristicFunction(new HashSet<string>(), chain).OrderBy(x=>x))}}}   (arguments sans attaquant = defendus par ∅)");
var f1 = CharacteristicFunction(new HashSet<string>(), chain);
sb.AppendLine($"F(F(∅)) = {{{string.Join(", ", CharacteristicFunction(f1, chain).OrderBy(x=>x))}}}  (A defense par ∅, C defense par A qui contre-attaque B)");
sb.ToString().Display();

Cadre : AF(args={A, B, C, D}, attacks={A->B, B->C, C->D})
F(∅)  = {A}   (arguments sans attaquant = defendus par ∅)
F(F(∅)) = {A, C}  (A defense par ∅, C defense par A qui contre-attaque B)


## 3. Ensembles sans conflit, admissibles, complets

- **Sans conflit** (conflict-free) : aucun element de $S$ n'en attaque un autre.
- **Admissible** : sans conflit ET chaque element de $S$ est defendu par $S$ ($S \subseteq F(S)$).
- **Complete** : admissible ET tout argument defendu par $S$ est dans $S$ ($F(S) \subseteq S$).

Un ensemble complet est donc un **point fixe local** de $F$ : defendre ce qu'il contient, et rien de plus.

In [4]:
// Sans conflit : aucun a, b dans S avec a -> b
static bool ConflictFree(ISet<string> S, DungAF af)
{
    foreach (var a in S)
        foreach (var b in S)
            if (af.Attacks(a, b)) return false;
    return true;
}

// Admissible : sans conflit ET tout a dans S est defendu par S
static bool IsAdmissible(ISet<string> S, DungAF af)
{
    if (!ConflictFree(S, af)) return false;
    foreach (var a in S)
        if (!DefendedBy(a, S, af)) return false;
    return true;
}

// Complet : admissible ET tout argument defendu par S est dans S
static bool IsComplete(ISet<string> S, DungAF af)
{
    if (!IsAdmissible(S, af)) return false;
    foreach (var a in af.Args)
        if (DefendedBy(a, S, af) && !S.Contains(a)) return false;
    return true;
}

// Enumere tous les sous-ensembles (powerset) — borne a |A| <= ~16 pour rester viable.
static List<HashSet<string>> AllSubsets(DungAF af)
{
    var args = af.Args.OrderBy(x => x).ToList();
    int n = args.Count;
    var result = new List<HashSet<string>>();
    for (int mask = 0; mask < (1 << n); mask++)
    {
        var s = new HashSet<string>();
        for (int i = 0; i < n; i++)
            if ((mask & (1 << i)) != 0) s.Add(args[i]);
        result.Add(s);
    }
    return result;
}

$"Briques semantiques pretes (ConflictFree / IsAdmissible / IsComplete / AllSubsets).".Display();

Briques semantiques pretes (ConflictFree / IsAdmissible / IsComplete / AllSubsets).

## 4. Semantique grounded : le plus petit point fixe

L'**extension grounded** est le **plus petit point fixe** de $F$, obtenu en iterant depuis $\emptyset$ :

$$S_0 = \emptyset,\quad S_{k+1} = F(S_k),\quad \text{jusqu'a } S_{k+1} = S_k$$

Cette suite est **croissante** ($S_k \subseteq S_{k+1}$) et **converge** en au plus $|A|$ etapes vers l'unique extension grounded. Elle est toujours definie (meme sur des AF cycliques) — c'est sa robustesse.

In [5]:
// Extension grounded = plus petit point fixe de F, itere depuis ∅.
static HashSet<string> Grounded(DungAF af)
{
    var S = new HashSet<string>();
    while (true)
    {
        var next = CharacteristicFunction(S, af);
        if (next.SetEquals(S)) return S;
        S = next;
    }
}

// Exemple : chaine A -> B -> C -> D.
var chain = new DungAF();
chain.AddAttack("A", "B"); chain.AddAttack("B", "C"); chain.AddAttack("C", "D");
var g = Grounded(chain);
var sb = new StringBuilder();
sb.AppendLine("=== Chaine A -> B -> C -> D ===");
sb.AppendLine($"AF : {chain}");
sb.AppendLine($"Extension grounded = {{{string.Join(", ", g.OrderBy(x => x))}}}");
sb.AppendLine("  (A : aucun attaquant, accepte. C : attaquant B contre-attaque par A. B et D exclus.)");
sb.ToString().Display();

=== Chaine A -> B -> C -> D ===
AF : AF(args={A, B, C, D}, attacks={A->B, B->C, C->D})
Extension grounded = {A, C}
  (A : aucun attaquant, accepte. C : attaquant B contre-attaque par A. B et D exclus.)


## 5. Semantiques stable et preferred

- **Stable** : sans conflit ET qui attaque **tout** argument hors de $S$. Peut **ne pas exister** (ex. cycle impair, auto-attaque non defendue). Quand elle existe, une extension stable est aussi complete et preferred.
- **Preferred** : extension admissible **maximale** par inclusion. Il en existe toujours au moins une (l'extension grounded elle-meme est admissible), mais elles peuvent etre **multiples** (ex. attaque mutuelle $a \leftrightarrow b$ : deux extensions preferred $\{a\}$ et $\{b\}$).

On les calcule par **enumeration de la powerset** (viable pour de petits AF pedagogiques).

In [6]:
// Stable : sans conflit ET attaque tout argument hors de S
static bool IsStable(ISet<string> S, DungAF af)
{
    if (!ConflictFree(S, af)) return false;
    foreach (var a in af.Args)
    {
        if (S.Contains(a)) continue;
        bool attacked = false;
        foreach (var b in S)
            if (af.Attacks(b, a)) { attacked = true; break; }
        if (!attacked) return false;
    }
    return true;
}

static List<HashSet<string>> StableExtensions(DungAF af)
    => AllSubsets(af).Where(s => IsStable(s, af)).ToList();

// Preferred = admissibles maximaux par inclusion
static List<HashSet<string>> PreferredExtensions(DungAF af)
{
    var adm = AllSubsets(af).Where(s => IsAdmissible(s, af)).ToList();
    var pref = new List<HashSet<string>>();
    foreach (var s in adm)
    {
        bool maximal = true;
        foreach (var other in adm)
        {
            if (other.Count > s.Count && s.IsSubsetOf(other)) { maximal = false; break; }
        }
        if (maximal) pref.Add(s);
    }
    return pref;
}

static List<HashSet<string>> CompleteExtensions(DungAF af)
    => AllSubsets(af).Where(s => IsComplete(s, af)).ToList();

static string Fmt(IEnumerable<HashSet<string>> exts) =>
    string.Join(", ", exts.Select(e => "{" + string.Join(", ", e.OrderBy(x => x)) + "}"));

// Attaque mutuelle a <-> b : DEUX extensions stables / preferred.
var mutual = new DungAF();
mutual.AddAttack("a", "b"); mutual.AddAttack("b", "a");
var sb = new StringBuilder();
sb.AppendLine("=== Attaque mutuelle a <-> b ===");
sb.AppendLine($"AF : {mutual}");
sb.AppendLine($"Grounded   = {{{string.Join(", ", Grounded(mutual).OrderBy(x=>x))}}}   (aucun des deux n'est defendu par ∅)");
sb.AppendLine($"Stable     = {Fmt(StableExtensions(mutual))}   (deux extensions : 'a' attaque b, ou l'inverse)");
sb.AppendLine($"Preferred  = {Fmt(PreferredExtensions(mutual))}");
sb.AppendLine($"Complete   = {Fmt(CompleteExtensions(mutual))}   (∅ inclus : complete trivial)");
sb.ToString().Display();

=== Attaque mutuelle a <-> b ===
AF : AF(args={a, b}, attacks={a->b, b->a})
Grounded   = {}   (aucun des deux n'est defendu par ∅)
Stable     = {a}, {b}   (deux extensions : 'a' attaque b, ou l'inverse)
Preferred  = {a}, {b}
Complete   = {}, {a}, {b}   (∅ inclus : complete trivial)


## 6. Cas limite : le cycle impair (pas d'extension stable)

Sur un **cycle de longueur impaire** $a \to b \to c \to a$, aucun sous-ensemble sans conflit n'attaque tous les autres : la **semantique stable n'existe pas**. C'est precisement ce qui motive les semantiques grounded/complete (qui, elles, existent toujours — ici grounded $= \emptyset$).

In [7]:
// Cycle a -> b -> c -> a : aucun attaquant non-mis-en-echec, grounded vide, PAS de stable.
var cyc3 = new DungAF();
cyc3.AddAttack("a", "b"); cyc3.AddAttack("b", "c"); cyc3.AddAttack("c", "a");
var sb = new StringBuilder();
sb.AppendLine("=== Cycle impair a -> b -> c -> a ===");
sb.AppendLine($"AF : {cyc3}");
sb.AppendLine($"Grounded   = {{{string.Join(", ", Grounded(cyc3).OrderBy(x=>x))}}}");
var stab = StableExtensions(cyc3);
sb.AppendLine($"Stable     = {(stab.Count == 0 ? "(aucune — le cycle impair n'a pas d'extension stable)" : Fmt(stab))}");
sb.AppendLine($"Preferred  = {Fmt(PreferredExtensions(cyc3))}   (∅ seul : rien n'est defendu)");
sb.AppendLine($"Complete   = {Fmt(CompleteExtensions(cyc3))}");
sb.AppendLine();
sb.AppendLine(">>> La semantique grounded (ici ∅) est TOUJOURS definie — sa robustesse face aux cycles.");
sb.ToString().Display();

=== Cycle impair a -> b -> c -> a ===
AF : AF(args={a, b, c}, attacks={a->b, b->c, c->a})
Grounded   = {}
Stable     = (aucune — le cycle impair n'a pas d'extension stable)
Preferred  = {}   (∅ seul : rien n'est defendu)
Complete   = {}

>>> La semantique grounded (ici ∅) est TOUJOURS definie — sa robustesse face aux cycles.


## 7. Labeling a trois valeurs (in / out / undec)

Une **labeling** $L = (\mathrm{in}, \mathrm{out}, \mathrm{undec})$ partitionne $A$ :
- $\mathrm{in}(L)$ : arguments **acceptes** (l'extension),
- $\mathrm{out}(L)$ : arguments **rejetes** (attaqus par un argument *in*),
- $\mathrm{undec}(L)$ : arguments **non decides** (ni accepts ni rejetes).

Les labelings completes sont en **bijection** avec les extensions completes : chaque extension complete correspond a exactement un labeling complet. Le labeling rend explicite le statut *undec* (indécis) que l'extension seule masque.

In [8]:
// Labeling 3-values associe a une extension : in = ext, out = attaques par in, undec = reste.
static (HashSet<string> In, HashSet<string> Out, HashSet<string> Undec) Labeling(ISet<string> ext, DungAF af)
{
    var inL = new HashSet<string>(ext);
    var outL = new HashSet<string>();
    foreach (var a in af.Args)
    {
        if (inL.Contains(a)) continue;
        foreach (var b in ext)
            if (af.Attacks(b, a)) { outL.Add(a); break; }
    }
    var undecL = new HashSet<string>(af.Args);
    undecL.ExceptWith(inL); undecL.ExceptWith(outL);
    return (inL, outL, undecL);
}

// Verif : chaine A->B->C->D, extension grounded {A, C}.
var chain = new DungAF();
chain.AddAttack("A", "B"); chain.AddAttack("B", "C"); chain.AddAttack("C", "D");
var g = Grounded(chain);
var lab = Labeling(g, chain);
var sb = new StringBuilder();
sb.AppendLine($"Chaine A->B->C->D, extension grounded = {{{string.Join(", ", g.OrderBy(x=>x))}}}");
sb.AppendLine($"  in    = {{{string.Join(", ", lab.In.OrderBy(x=>x))}}}   (acceptes)");
sb.AppendLine($"  out   = {{{string.Join(", ", lab.Out.OrderBy(x=>x))}}}  (rejetes : attaques par A ou C)");
sb.AppendLine($"  undec = {{{string.Join(", ", lab.Undec.OrderBy(x=>x))}}} (indécis)");
sb.AppendLine();
sb.AppendLine(">>> B et D sont 'out' (attaqus), aucun 'undec' ici. Sur le cycle impair, tout serait 'undec'.");
sb.ToString().Display();

Chaine A->B->C->D, extension grounded = {A, C}
  in    = {A, C}   (acceptes)
  out   = {B, D}  (rejetes : attaques par A ou C)
  undec = {} (indécis)

>>> B et D sont 'out' (attaqus), aucun 'undec' ici. Sur le cycle impair, tout serait 'undec'.


## 8. Generation aleatoire de cadres (cf. Python §4.1.3)

Pour etudier statistiquement les semantiques, on genere des AF aleatoires : $n$ arguments, chaque attaque $a \to b$ ($a \neq b$) presente avec probabilite $p$. La **densite** $p$ controle la connectivite du graphe d'attaque.

A basse densite, peu d'attaques : l'extension grounded est souvent grande (arguments isoles non attaques). A haute densite, l'attaque generalisee fait chuter la taille grounded (rien n'est defendu).

In [9]:
// AF aleatoire : n arguments, chaque attaque a->b (a != b) presente avec probabilite p.
static DungAF RandomAF(int n, double p, Random rng)
{
    var af = new DungAF();
    for (int i = 0; i < n; i++) af.AddArg($"A{i}");
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
        {
            if (i == j) continue;  // pas d'auto-attaque ici (modele Erdos-Renyi simple)
            if (rng.NextDouble() < p) af.AddAttack($"A{i}", $"A{j}");
        }
    return af;
}

// Etude : taille moyenne de l'extension grounded selon la densite (n=6, 200 tirages chacune).
var sb = new StringBuilder();
sb.AppendLine("=== Taille grounded vs densite (n=6 args, 200 AF aleatoires) ===");
sb.AppendLine($"{"p",6} {"grounded moyen",16} {"AF sans stable",16}");
foreach (var p in new[] { 0.1, 0.3, 0.5, 0.7, 0.9 })
{
    double sumG = 0; int noStable = 0; const int TRIALS = 200;
    for (int t = 0; t < TRIALS; t++)
    {
        var af = RandomAF(6, p, rng);
        sumG += Grounded(af).Count;
        if (StableExtensions(af).Count == 0) noStable++;
    }
    sb.AppendLine($"{p,6:F1} {sumG / TRIALS,16:F2} {(double)noStable / TRIALS,16:P0}");
}
sb.AppendLine();
sb.AppendLine(">>> A p=0.1 (peu d'attaques) la grounded est large ; a p=0.9 elle s'effondre.");
sb.AppendLine(">>> La proportion d'AF sans extension stable croit avec la densite (cycles impairs).");
sb.ToString().Display();

=== Taille grounded vs densite (n=6 args, 200 AF aleatoires) ===
     p   grounded moyen   AF sans stable
   0,1             4,03              1 %
   0,3             1,75              6 %
   0,5             0,28              7 %
   0,7             0,02              6 %
   0,9             0,00              0 %

>>> A p=0.1 (peu d'attaques) la grounded est large ; a p=0.9 elle s'effondre.
>>> La proportion d'AF sans extension stable croit avec la densite (cycles impairs).


### Exercice 1 : Enumeration des extensions stables

Soit l'AF : $a \to b$, $b \to c$, $c \to b$, $d \to a$, $a \to d$ (les arguments $a$ et $d$ s'attaquent mutuellement ; $b$ et $c$ forment un cycle). Completez le stub pour enumerer et afficher **toutes** les extensions stables et preferred de cet AF. Combien y en a-t-il ?

In [10]:
// Exercice 1 : enumerer stable + preferred d'un AF a 4 arguments (etudiant a completer)
// Indice 1 : construire l'AF avec AddAttack, appeler StableExtensions / PreferredExtensions.
// Indice 2 : Fmt(...) donne une representation texte d'une liste d'extensions.
// Etape 1 : construire l'AF (a->b, b->c, c->b, d->a, a->d)
// Etape 2 : afficher StableExtensions(af) et PreferredExtensions(af)
// TODO etudiant : completer
"Exercice 1 a completer — enumerer stable + preferred de l'AF a 4 arguments.".Display();

Exercice 1 a completer — enumerer stable + preferred de l'AF a 4 arguments.

### Exercice 2 : Grounded vs complete

Sur le AF de l'Exercice 1, comparez l'extension **grounded** (plus petit point fixe) et les extensions **completes**. L'extension grounded est-elle toujours complete ? Est-elle toujours la plus petite extension complete ? Verifiez avec `Grounded(af)` et `CompleteExtensions(af)`.

In [11]:
// Exercice 2 : comparer grounded et extensions completes (etudiant a completer)
// Indice : Grounded(af) retourne un HashSet ; CompleteExtensions(af) une List<HashSet>.
// Verifier : grounded ⊆ chaque complete ; grounded est lui-meme complete ?
// TODO etudiant : completer
"Exercice 2 a completer — grounded vs extensions completes.".Display();

Exercice 2 a completer — grounded vs extensions completes.

### Exercice 3 : Densite et extinctions

Reprenez l'etude statistique de la §8 en faisant varier $n \in \{4, 6, 8, 10\}$ a densite fixe $p = 0.5$. La **proportion d'AF sans extension stable** augmente-t-elle avec $n$ ? La taille grounded moyenne (normalisee par $n$) diminue-t-elle ? Interpretez : pourquoi un grand AF dense n'admet-il plus de point fixe stable ?

In [12]:
// Exercice 3 : impact de la taille n sur la grounded et l'existence de stable (etudiant a completer)
// Indice : pour chaque n in {4,6,8,10}, boucler TRIALS tirages a p=0.5, collecter sumG/TRIALS et noStable.
// TODO etudiant : completer le sweep
double[] ns = { 4, 6, 8, 10 };
"Exercice 3 a completer — sweep n in {4,6,8,10} a p=0.5.".Display();

Exercice 3 a completer — sweep n in {4,6,8,10} a p=0.5.

## 9. Conclusion (tranche 1)

Cette tranche a pose le **moteur Dung from-scratch** en C#/.NET : fonction caracteristique, semantiques grounded/stable/preferred/complete, labeling 3-valued, generation aleatoire.

### Recapitulatif

| Concept | Implementation C# |
|---------|-------------------|
| Cadre d'argumentation (AF) | `DungAF` (args + `attackersOf`/`attackedBy` indexes) |
| Fonction caracteristique $F(S)$ | `CharacteristicFunction` (acceptabilite = contre-attaque) |
| Admissible / Complet | `IsAdmissible` / `IsComplete` (powerset enum) |
| Grounded | `Grounded` (iteration du point fixe depuis $\emptyset$) |
| Stable / Preferred | `IsStable` / `PreferredExtensions` (maximaux par inclusion) |
| Labeling 3-values | `Labeling` (in / out / undec) — bijection avec complet |
| AF aleatoire | `RandomAF` (Erdos-Renyi, densite $p$) |

### Points cles

1. **Complementarite from-scratch ↔ librairie** : le twin Python + Tweety-3-Dung-Csharp (IKVM) montrent la **lib Tweety** (solveur tout pret, semantiques avancees CF2/equivalence), ce twin .NET demontre les semantiques **from-scratch** (comprendre $F$ et l'enumeration). Deux angles pedagogiques complementaires (#3801 Prong B).
2. **Grounded = robuste** : toujours defini (meme sur cycles impairs ou auto-attaques), car plus petit point fixe de $F$ itere depuis $\emptyset$. Stable, lui, peut **ne pas exister**.
3. **Bijection labeling ↔ complet** : chaque extension complete correspond a un unique labeling 3-values. Le statut *undec* explicite ce que l'extension masque.

### Tranches suivantes (marathon #4956)

- **Tranche 2** : semantiques avancees via la lib Tweety (CF2 pour cycles, equivalence de frameworks, explications) — angle librairie.
- **Tranche 3** : apprentissage de cadres depuis des labelings (inférer la relation d'attaque).

### References

- Dung, P.M. (1995). *On the Acceptability of Arguments and its Fundamental Role in Nonmonotonic Reasoning, Logic Programming, and n-Person Games*. Artificial Intelligence 77.
- Twin Python : [Tweety-5-Abstract-Argumentation](Tweety-5-Abstract-Argumentation.ipynb) (Tweety-JVM).
- Twin .NET IKVM : [Tweety-3-Dung-Csharp](Tweety-3-Dung-Csharp.ipynb).

---

*Tranche 1 du twin .NET⇄Python (#4956 marathon). Argumentation de Dung = 1er concept (apres les logiques de Tweety-2/3). from-scratch = la valeur pedagogique de ce twin.*